# Assignment 2 - Feedforward NN (MNIST)
**Offline version** - runs without internet.
Needs `mnist.npz` in the same folder (see `00_download_once.ipynb`).

In [ ]:
# Assignment 2 - Feedforward neural network (MNIST) with Keras / TensorFlow, SGD
import numpy as np, matplotlib

# ---- OFFLINE MNIST LOADER (no internet needed) --------------------------------
# Reads mnist.npz from the notebook folder (or from ~/.keras/datasets).
# Get the file once from: https://storage.googleapis.com/tensorflow/tf-keras-datasets/mnist.npz
import os
def load_mnist(path="mnist.npz"):
    for p in (path, os.path.expanduser("~/.keras/datasets/mnist.npz")):
        if os.path.exists(p):
            with np.load(p, allow_pickle=True) as f:
                return (f["x_train"], f["y_train"]), (f["x_test"], f["y_test"])
    raise FileNotFoundError("mnist.npz not found. Put it in the same folder as this notebook "
        "(download link: https://storage.googleapis.com/tensorflow/tf-keras-datasets/mnist.npz)")
# -------------------------------------------------------------------------------

#matplotlib.use("Agg")
%matplotlib inline
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import classification_report
tf.random.set_seed(42); np.random.seed(42)

(x_train, y_train), (x_test, y_test) = load_mnist()
x_train = x_train.reshape(-1, 784).astype("float32") / 255.0
x_test = x_test.reshape(-1, 784).astype("float32") / 255.0
print("Training data:", x_train.shape, " Testing data:", x_test.shape)

model = keras.Sequential([
    layers.Input((784,)),
    layers.Dense(256, activation="sigmoid"),
    layers.Dense(128, activation="sigmoid"),
    layers.Dense(10, activation="softmax"),
])
model.summary()
model.compile(optimizer=keras.optimizers.SGD(learning_rate=0.01),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
H = model.fit(x_train, y_train, validation_data=(x_test, y_test), epochs=100, batch_size=128,
    verbose=2)

loss, acc = model.evaluate(x_test, y_test, verbose=0)
print(f"\nTest loss: {loss:.4f}   Test accuracy: {acc:.4f}")
pred = model.predict(x_test, verbose=0).argmax(axis=1)
print(classification_report(y_test, pred, digits=4))

plt.style.use("default")
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(H.history["loss"], label="train_loss")
ax[0].plot(H.history["val_loss"], label="val_loss")
ax[0].set_title("Training and Validation Loss")
ax[0].set_xlabel("Epoch")
ax[0].set_ylabel("Loss")
ax[0].legend()
ax[0].grid(alpha=.3)
ax[1].plot(H.history["accuracy"], label="train_acc")
ax[1].plot(H.history["val_accuracy"], label="val_acc")
ax[1].set_title("Training and Validation Accuracy")
ax[1].set_xlabel("Epoch")
ax[1].set_ylabel("Accuracy")
ax[1].legend()
ax[1].grid(alpha=.3)
#plt.tight_layout(); plt.savefig("a2_curves.png", dpi=130)
plt.tight_layout()
plt.savefig("a2_curves.png", dpi=130)   # optional: keeps a copy as a file
plt.show()                              # this line displays the graph